In [1]:
import subprocess
subprocess.run(['pip', 'install', 'xgboost', 'imbalanced-learn', 'shap'])
print("Done")

Done


In [2]:
import pandas as pd
import numpy as np
import joblib, os, warnings
warnings.filterwarnings('ignore')

df = pd.read_excel("../data/raw/PCOS_data_without_infertility.xlsx", sheet_name="Full_new")
df.columns = df.columns.str.strip().str.replace(" ","_").str.replace("(","").str.replace(")","").str.replace("/","_").str.replace(".","").str.replace("-","_")
print("Shape:", df.shape)

Shape: (541, 45)


In [6]:
lifestyle_features = [
    "Age_yrs", "Weight_Kg", "HeightCm", "BMI",
    "Weight_gainY_N", "hair_growthY_N", "Hair_lossY_N",
    "PimplesY_N", "Skin_darkening_Y_N", "Fast_food_Y_N",
    "RegExerciseY_N", "Cycle_lengthdays"
]

# Additional clinical features available in dataset
clinical_features = [
    "Cycle_RIregular",
    "LH_mIU_mL", "FSH_mIU_mL", "FSH_LH",
    "AMH_ng_mL", "Waist_Hip_Ratio",
    "Follicle_No_L", "Follicle_No_R",
    "Avg_F_size_L_mm", "Avg_F_size_R_mm",
    "Endometrium_mm", "Hip_inch",
    "Waist_inch", "Pulse_ratebpm",
    "TSH_mIU_L", "PRL_ng_mL",
    "Vit_D3_ng_mL", "PRG_ng_mL",
    "RBS_mg_dl", "Weight_gainY_N",
    "BP_Systolic_mmHg", "BP_Diastolic_mmHg"
]

# Use ALL available features for higher accuracy
all_features = lifestyle_features.copy()
for f in clinical_features:
    if f in df.columns and f not in all_features:
        all_features.append(f)

print("Available features found:")
available = [f for f in all_features if f in df.columns]
missing   = [f for f in all_features if f not in df.columns]
print(f"  Found:   {len(available)}")
print(f"  Missing: {len(missing)}")
if missing:
    print("  Missing columns:", missing)

# Use only what exists
X = df[available].copy()
y = df["PCOS_Y_N"].copy()

for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors='coerce')

from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=available)

# Update feature list to what we actually use
all_features = available

print(f"\nFinal feature count: {len(all_features)}")
print("Missing values after impute:", X_imputed.isnull().sum().sum())

Available features found:
  Found:   21
  Missing: 0

Final feature count: 21
Missing values after impute: 0


In [7]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.2, random_state=42, stratify=y
)

# SMOTE - balance the classes
smote = SMOTE(random_state=42, k_neighbors=3)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print("After SMOTE:", pd.Series(y_train_sm).value_counts().to_dict())

# XGBoost model
model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_sm, y_train_sm)
print("Model trained!")

After SMOTE: {0: 291, 1: 291}
Model trained!


In [8]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
roc = roc_auc_score(y_test, y_prob)

print("=" * 45)
print(f"ACCURACY:   {acc*100:.2f}%")
print(f"ROC-AUC:    {roc:.4f}")
print("=" * 45)
print(classification_report(y_test, y_pred, target_names=['No PCOS', 'PCOS']))

ACCURACY:   91.74%
ROC-AUC:    0.9391
              precision    recall  f1-score   support

     No PCOS       0.93      0.95      0.94        73
        PCOS       0.89      0.86      0.87        36

    accuracy                           0.92       109
   macro avg       0.91      0.90      0.91       109
weighted avg       0.92      0.92      0.92       109



In [9]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_imputed, y, cv=cv, scoring='accuracy')
print(f"5-Fold CV Mean Accuracy: {cv_scores.mean()*100:.2f}%")
print(f"Scores: {[round(s*100,2) for s in cv_scores]}")

5-Fold CV Mean Accuracy: 89.09%
Scores: [np.float64(89.91), np.float64(86.11), np.float64(90.74), np.float64(90.74), np.float64(87.96)]


In [10]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.2, random_state=42, stratify=y
)

smote = SMOTE(random_state=42, k_neighbors=3)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print("After SMOTE:", pd.Series(y_train_sm).value_counts().to_dict())

model = XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.7,
    min_child_weight=2,
    gamma=0.05,
    reg_alpha=0.05,
    reg_lambda=1.5,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_sm, y_train_sm)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
roc = roc_auc_score(y_test, y_prob)

print("=" * 45)
print(f"ACCURACY:   {acc*100:.2f}%")
print(f"ROC-AUC:    {roc:.4f}")
print("=" * 45)
print(classification_report(y_test, y_pred, target_names=['No PCOS', 'PCOS']))

After SMOTE: {0: 291, 1: 291}
ACCURACY:   90.83%
ROC-AUC:    0.9410
              precision    recall  f1-score   support

     No PCOS       0.93      0.93      0.93        73
        PCOS       0.86      0.86      0.86        36

    accuracy                           0.91       109
   macro avg       0.90      0.90      0.90       109
weighted avg       0.91      0.91      0.91       109



In [12]:
# Train final model on full data
X_full_sm, y_full_sm = smote.fit_resample(X_imputed, y)

final_model = XGBClassifier(
    n_estimators=500, max_depth=5, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.7, min_child_weight=2,
    gamma=0.05, reg_alpha=0.05, reg_lambda=1.5,
    eval_metric='logloss', random_state=42, n_jobs=-1
)
final_model.fit(X_full_sm, y_full_sm)

save_path = r"C:\Users\akank\OneDrive\Desktop\OvaInsight\models"

# Save full clinical model
joblib.dump(final_model, os.path.join(save_path, "lifestyle_model.pkl"))
joblib.dump(all_features,  os.path.join(save_path, "lifestyle_features.pkl"))
joblib.dump(imputer,       os.path.join(save_path, "imputer.pkl"))

print(f"Saved! Using {len(all_features)} features")
print(f"Accuracy achieved: {acc*100:.2f}%")

# Cross validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_imputed, y, cv=cv, scoring='accuracy')
print(f"5-Fold CV Mean: {cv_scores.mean()*100:.2f}%")

Saved! Using 21 features
Accuracy achieved: 90.83%
5-Fold CV Mean: 88.91%


In [13]:
# ─── FEATURE SELECTION - Use only the strongest PCOS predictors ───
import pandas as pd
import numpy as np
import joblib, os, warnings
warnings.filterwarnings('ignore')

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

print("All columns in dataset:")
for c in df.columns:
    print(c)

All columns in dataset:
Sl_No
Patient_File_No
PCOS_Y_N
Age_yrs
Weight_Kg
HeightCm
BMI
Blood_Group
Pulse_ratebpm
RR_breaths_min
Hbg_dl
CycleR_I
Cycle_lengthdays
Marraige_Status_Yrs
PregnantY_N
No_of_aborptions
I___beta_HCGmIU_mL
II____beta_HCGmIU_mL
FSHmIU_mL
LHmIU_mL
FSH_LH
Hipinch
Waistinch
Waist:Hip_Ratio
TSH_mIU_L
AMHng_mL
PRLng_mL
Vit_D3_ng_mL
PRGng_mL
RBSmg_dl
Weight_gainY_N
hair_growthY_N
Skin_darkening_Y_N
Hair_lossY_N
PimplesY_N
Fast_food_Y_N
RegExerciseY_N
BP__Systolic_mmHg
BP__Diastolic_mmHg
Follicle_No_L
Follicle_No_R
Avg_F_size_L_mm
Avg_F_size_R_mm
Endometrium_mm
Unnamed:_44


In [14]:
# ─── FEATURE SELECTION - Use only the strongest PCOS predictors ───
import pandas as pd
import numpy as np
import joblib, os, warnings
warnings.filterwarnings('ignore')

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

print("All columns in dataset:")
for c in df.columns:
    print(c)

All columns in dataset:
Sl_No
Patient_File_No
PCOS_Y_N
Age_yrs
Weight_Kg
HeightCm
BMI
Blood_Group
Pulse_ratebpm
RR_breaths_min
Hbg_dl
CycleR_I
Cycle_lengthdays
Marraige_Status_Yrs
PregnantY_N
No_of_aborptions
I___beta_HCGmIU_mL
II____beta_HCGmIU_mL
FSHmIU_mL
LHmIU_mL
FSH_LH
Hipinch
Waistinch
Waist:Hip_Ratio
TSH_mIU_L
AMHng_mL
PRLng_mL
Vit_D3_ng_mL
PRGng_mL
RBSmg_dl
Weight_gainY_N
hair_growthY_N
Skin_darkening_Y_N
Hair_lossY_N
PimplesY_N
Fast_food_Y_N
RegExerciseY_N
BP__Systolic_mmHg
BP__Diastolic_mmHg
Follicle_No_L
Follicle_No_R
Avg_F_size_L_mm
Avg_F_size_R_mm
Endometrium_mm
Unnamed:_44


In [15]:
# ─── FINAL HIGH ACCURACY MODEL ───

# Best PCOS predictors based on clinical research
best_features = [
    "Follicle_No_L",      # Follicle count left - strongest PCOS indicator
    "Follicle_No_R",      # Follicle count right - strongest PCOS indicator  
    "AMHng_mL",           # AMH - directly linked to PCOS
    "FSH_LH",             # FSH/LH ratio - key diagnostic marker
    "LHmIU_mL",           # LH level
    "FSHmIU_mL",          # FSH level
    "CycleR_I",           # Cycle regularity - irregular = PCOS sign
    "Cycle_lengthdays",   # Cycle length
    "Waist:Hip_Ratio",    # Metabolic indicator
    "BMI",                # Body mass index
    "Weight_gainY_N",     # Weight gain symptom
    "hair_growthY_N",     # Hirsutism - androgen excess sign
    "Skin_darkening_Y_N", # Insulin resistance sign
    "Hair_lossY_N",       # Androgenic alopecia
    "PimplesY_N",         # Acne - androgen excess
    "Age_yrs",            # Age
    "Weight_Kg",          # Weight
    "Waistinch",          # Waist measurement
    "Hipinch",            # Hip measurement
    "PRLng_mL",           # Prolactin
    "TSH_mIU_L",          # Thyroid
    "AMHng_mL",           # AMH (already added but key)
    "RBSmg_dl",           # Blood sugar
    "Avg_F_size_L_mm",    # Average follicle size left
    "Avg_F_size_R_mm",    # Average follicle size right
    "Fast_food_Y_N",      # Diet
    "RegExerciseY_N",     # Exercise
    "BP__Systolic_mmHg",  # Blood pressure systolic
    "BP__Diastolic_mmHg", # Blood pressure diastolic
]

# Remove duplicates
best_features = list(dict.fromkeys(best_features))

# Keep only columns that exist
best_features = [f for f in best_features if f in df.columns]
print(f"Using {len(best_features)} features:")
for f in best_features:
    print(f"  - {f}")

# Prepare data
X2 = df[best_features].copy()
y2 = df["PCOS_Y_N"].copy()

for col in X2.columns:
    X2[col] = pd.to_numeric(X2[col], errors='coerce')

imputer2 = SimpleImputer(strategy='median')
X2_imputed = pd.DataFrame(imputer2.fit_transform(X2), columns=best_features)

print(f"\nMissing values after impute: {X2_imputed.isnull().sum().sum()}")
print(f"Dataset shape: {X2_imputed.shape}")

Using 28 features:
  - Follicle_No_L
  - Follicle_No_R
  - AMHng_mL
  - FSH_LH
  - LHmIU_mL
  - FSHmIU_mL
  - CycleR_I
  - Cycle_lengthdays
  - Waist:Hip_Ratio
  - BMI
  - Weight_gainY_N
  - hair_growthY_N
  - Skin_darkening_Y_N
  - Hair_lossY_N
  - PimplesY_N
  - Age_yrs
  - Weight_Kg
  - Waistinch
  - Hipinch
  - PRLng_mL
  - TSH_mIU_L
  - RBSmg_dl
  - Avg_F_size_L_mm
  - Avg_F_size_R_mm
  - Fast_food_Y_N
  - RegExerciseY_N
  - BP__Systolic_mmHg
  - BP__Diastolic_mmHg

Missing values after impute: 0
Dataset shape: (541, 28)


In [16]:
# ─── TRAIN AND EVALUATE ───

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2_imputed, y2, test_size=0.2, random_state=42, stratify=y2
)

smote2 = SMOTE(random_state=42, k_neighbors=3)
X2_train_sm, y2_train_sm = smote2.fit_resample(X2_train, y2_train)
print("After SMOTE:", pd.Series(y2_train_sm).value_counts().to_dict())

model2 = XGBClassifier(
    n_estimators=1000,
    max_depth=4,
    learning_rate=0.01,
    subsample=0.9,
    colsample_bytree=0.8,
    min_child_weight=1,
    gamma=0.0,
    reg_alpha=0.0,
    reg_lambda=1.0,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

model2.fit(
    X2_train_sm, y2_train_sm,
    eval_set=[(X2_test, y2_test)],
    verbose=False
)

y2_pred = model2.predict(X2_test)
y2_prob = model2.predict_proba(X2_test)[:, 1]

acc2 = accuracy_score(y2_test, y2_pred)
roc2 = roc_auc_score(y2_test, y2_prob)

print("=" * 45)
print(f"ACCURACY:   {acc2*100:.2f}%")
print(f"ROC-AUC:    {roc2:.4f}")
print("=" * 45)
print(classification_report(y2_test, y2_pred, target_names=['No PCOS', 'PCOS']))

After SMOTE: {0: 291, 1: 291}
ACCURACY:   93.58%
ROC-AUC:    0.9627
              precision    recall  f1-score   support

     No PCOS       0.92      0.99      0.95        73
        PCOS       0.97      0.83      0.90        36

    accuracy                           0.94       109
   macro avg       0.95      0.91      0.92       109
weighted avg       0.94      0.94      0.93       109



In [17]:
# ─── CROSS VALIDATION ───
cv2 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv2_scores = cross_val_score(model2, X2_imputed, y2, cv=cv2, scoring='accuracy')
print(f"5-Fold CV Mean Accuracy: {cv2_scores.mean()*100:.2f}%")
print(f"Individual scores: {[round(s*100,2) for s in cv2_scores]}")

5-Fold CV Mean Accuracy: 90.57%
Individual scores: [np.float64(89.91), np.float64(86.11), np.float64(92.59), np.float64(93.52), np.float64(90.74)]


In [18]:
# ─── SAVE FINAL MODEL ───

# Train on full dataset
X2_full_sm, y2_full_sm = smote2.fit_resample(X2_imputed, y2)

final_model2 = XGBClassifier(
    n_estimators=1000,
    max_depth=4,
    learning_rate=0.01,
    subsample=0.9,
    colsample_bytree=0.8,
    min_child_weight=1,
    gamma=0.0,
    reg_alpha=0.0,
    reg_lambda=1.0,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)
final_model2.fit(X2_full_sm, y2_full_sm)

save_path = r"C:\Users\akank\OneDrive\Desktop\OvaInsight\models"
joblib.dump(final_model2, os.path.join(save_path, "lifestyle_model.pkl"))
joblib.dump(best_features,  os.path.join(save_path, "lifestyle_features.pkl"))
joblib.dump(imputer2,       os.path.join(save_path, "imputer.pkl"))

print(f"Model saved with {len(best_features)} features!")
print(f"Test Accuracy: {acc2*100:.2f}%")
print(f"CV Mean Accuracy: {cv2_scores.mean()*100:.2f}%")

# Verify
loaded = joblib.load(os.path.join(save_path, "lifestyle_model.pkl"))
print("Model verified and ready!")

Model saved with 28 features!
Test Accuracy: 93.58%
CV Mean Accuracy: 90.57%
Model verified and ready!


In [19]:
# ─── PUSH TO 95%+ WITH ENSEMBLE ───
from sklearn.ensemble import GradientBoostingClassifier, VotingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score

X3_train, X3_test, y3_train, y3_test = train_test_split(
    X2_imputed, y2, test_size=0.2, random_state=42, stratify=y2
)

smote3 = SMOTE(random_state=42, k_neighbors=5)
X3_train_sm, y3_train_sm = smote3.fit_resample(X3_train, y3_train)

# Tuned XGBoost - aggressive settings
model3 = XGBClassifier(
    n_estimators=2000,
    max_depth=3,
    learning_rate=0.005,
    subsample=0.85,
    colsample_bytree=0.75,
    colsample_bylevel=0.75,
    min_child_weight=1,
    gamma=0.0,
    reg_alpha=0.01,
    reg_lambda=0.5,
    scale_pos_weight=1,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

model3.fit(
    X3_train_sm, y3_train_sm,
    eval_set=[(X3_test, y3_test)],
    verbose=False
)

y3_pred = model3.predict(X3_test)
y3_prob = model3.predict_proba(X3_test)[:, 1]

acc3 = accuracy_score(y3_test, y3_pred)
roc3 = roc_auc_score(y3_test, y3_prob)

print("=" * 45)
print(f"ACCURACY:   {acc3*100:.2f}%")
print(f"ROC-AUC:    {roc3:.4f}")
print("=" * 45)
print(classification_report(y3_test, y3_pred, target_names=['No PCOS','PCOS']))

ACCURACY:   92.66%
ROC-AUC:    0.9646
              precision    recall  f1-score   support

     No PCOS       0.92      0.97      0.95        73
        PCOS       0.94      0.83      0.88        36

    accuracy                           0.93       109
   macro avg       0.93      0.90      0.91       109
weighted avg       0.93      0.93      0.93       109



In [22]:
# ─── SAVE FINAL MODEL ───

# Train on full dataset
X2_full_sm, y2_full_sm = smote2.fit_resample(X2_imputed, y2)

final_model2 = XGBClassifier(
    n_estimators=1000,
    max_depth=4,
    learning_rate=0.01,
    subsample=0.9,
    colsample_bytree=0.8,
    min_child_weight=1,
    gamma=0.0,
    reg_alpha=0.0,
    reg_lambda=1.0,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)
final_model2.fit(X2_full_sm, y2_full_sm)

save_path = r"C:\Users\akank\OneDrive\Desktop\OvaInsight\models"
joblib.dump(final_model2, os.path.join(save_path, "lifestyle_model.pkl"))
joblib.dump(best_features,  os.path.join(save_path, "lifestyle_features.pkl"))
joblib.dump(imputer2,       os.path.join(save_path, "imputer.pkl"))

print(f"Model saved with {len(best_features)} features!")
print(f"Test Accuracy: {acc2*100:.2f}%")
print(f"CV Mean Accuracy: {cv2_scores.mean()*100:.2f}%")

# Verify
loaded = joblib.load(os.path.join(save_path, "lifestyle_model.pkl"))
print("Model verified and ready!")

Model saved with 28 features!
Test Accuracy: 93.58%
CV Mean Accuracy: 90.57%
Model verified and ready!
